# 01 — Qualidade dos dados (rascunho)

Fonte: `relatorios/perfil_dados.md`, gerado em 2026-10-22 por `python -m src.perfil_dados`, que só produz metadados agregados.
Este notebook **não guarda saídas**: ele traz contagens e percentuais, sem volumes em m², custos ou valores individuais.
Para ver o painel preparado, rode a célula de código do fim. Ela devolve apenas agregados.

## 1. O que chegou

São 6 arquivos e 13 tabelas. Nenhum deu erro de leitura, nenhum tem linhas duplicadas e nenhum tem linhas de total.
Todos os arquivos já vieram **tratados e agregados por SKU × mês**: são saída de um tratamento anterior, feito fora deste repositório. Não são exportação bruta do sistema.
Por isso, a cadeia reproduzível deste projeto começa nessas tabelas.

| Tabela | Papel | Uso no projeto |
|---|---|---|
| `vendas_estoque_mensal.csv` | **VENDAS (alvo)**, com compras e estoque reconstruído do mês | alvo = `vendas_m2`; SKU = `codigo_demanda`; mês = `mes` |
| `produtos.csv` | cadastro (linha, marca, acabamento, largura, m² por bobina, incluir/excluir, código de demanda) | filtro `incluir`; atributos para modelos futuros |
| xlsx `[estoque_atual]` | foto do estoque em 2026-10-20 | decisão a jusante (estoque inicial), não entra no backtest |
| `chegadas_por_item.csv`, xlsx `[containers]` | pedidos de importação em trânsito e chegadas previstas | decisão a jusante; lead time observado entre 4 e 5 meses, coerente com H = 5 |
| xlsx `[containers_financeiro]` | pagamentos das importações | irrelevante para a demanda |
| `baseline_cobertura.csv`, `pedido_sugerido.csv` | **saídas da regra atual** na posição de out/2026 | **não usar como entrada** (circular e calculadas com dados posteriores); servem só para conferir o baseline operacional |
| `vinil_dados_tratados.xlsx` (abas repetidas) | cópia das CSVs, com mesmas linhas e colunas | usamos as CSVs; do xlsx, só as abas que não existem em CSV |

## 2. Período e cobertura

| Item | Valor | Decisão |
|---|---|---|
| Período | 2026-02 a 2026-10 (9 meses) | — |
| Meses parciais | 2 por SKU (primeiro e último, coluna `mes_parcial`) | **excluídos**: meio mês de venda não é demanda mensal |
| Meses completos | **7** (2026-03 a 2026-09) | — |
| Meses sem nenhum registro | nenhum (todos os 73 códigos têm os 9 meses, 73 × 9 = 657 linhas) | — |
| ≥ 24 meses para o sazonal ingênuo? | **não** | sazonal ingênuo desligado automaticamente |

**Isso bloqueia o backtest da regra operacional.** Prever o mês *t* com informação até *t − 5* e janela de 6 meses exige 6 meses completos antes de *t − 5*.
O primeiro mês previsível seria o 11º mês completo, e a base tem 7. Com a regra exata, `python -m src.main --fonte real` avalia **0 linhas**.
Relaxando a janela para "os meses que houver" (`--min-meses-janela 1`), sobram só 2 meses de teste, e neles a janela tem 1 ou 2 meses de dados. Isso serve como diagnóstico, não como evidência.

Histórico mínimo para um backtest com H = 5, janela 6 e 6 meses de teste: 5 + 6 + 6 = **17 meses completos**. Para o sazonal ingênuo, **24 meses**.

## 3. Códigos e chaves

**Termos:** *código* = SKU do cadastro (`codigo`). *Código de demanda* = grupo de códigos equivalentes (`codigo_demanda`), que é a unidade de análise.

| Item | Valor | Decisão |
|---|---|---|
| Códigos no cadastro e nas vendas | 73 (casamento de 100% nos dois sentidos) | — |
| Códigos de demanda (equivalentes somados) | 64; todos existem como código no cadastro | unidade de análise = **`codigo_demanda`**, porque a venda migra entre equivalentes |
| Códigos marcados `incluir = falso` | 19 de 73 códigos (171 linhas); cada um é seu próprio código de demanda, então saem 19 de 64 códigos de demanda | **removidos**. A marcação vem do tratamento anterior (coluna `motivo_exclusao`, 5 motivos), mas **quem define e com qual critério não está documentado** (ver pendências) |
| Códigos de demanda incluídos | 45 | — |
| Códigos de demanda sem nenhuma venda nos 7 meses completos | 15 | **removidos do painel**; não há demanda observável para prever |
| **Códigos de demanda no painel** | **30** (204 linhas código de demanda × mês) | — |
| Itens em trânsito × cadastro | 30 códigos, todos no cadastro | — |

## 4. Problemas de qualidade e decisões

| # | Problema | Evidência (perfil) | Decisão | Por quê |
|---|---|---|---|---|
| 1 | Histórico curto | 7 meses completos | rodar os baselines só como diagnóstico; **pedir histórico mais longo à empresa** | o backtest com H = 5 precisa de ≥ 17 meses |
| 2 | Meses parciais | 2 por SKU | excluir | subestimam a demanda do mês |
| 3 | Nulos | 0% em todas as colunas da tabela de vendas | nada a fazer no alvo | nulos só aparecem em tabelas que não entram no backtest (ex.: curva ABC 100% vazia, câmbio médio 48%, data da última compra 23%) |
| 4 | Devoluções / negativos | 0 valores negativos em `vendas_m2` | sem tratamento; `TRATAMENTO_NEGATIVOS` fica só para a amostra | **pergunta à empresa:** a venda já vem líquida de devoluções ou as devoluções não foram exportadas? |
| 5 | Meses sem venda | 58% das linhas brutas com venda 0; no painel final, 10,3% | zeros mantidos a partir da 1ª venda de cada SKU | zero depois do lançamento é demanda real (ou ruptura); zero antes do lançamento seria falso |
| 6 | Ruptura | `possivel_ruptura` = 1 em 51% das linhas brutas; no painel final (SKU de demanda em que **todos** os códigos estavam em ruptura), 15,2% | manter no treino, marcar e **reportar as métricas também sem ruptura** (`sku_mes_sem_ruptura`) | na ruptura, a venda é menor que a demanda; se o código equivalente tinha estoque, a demanda foi para ele e a soma continua válida |
| 7 | Estoque reconstruído negativo | 108 linhas com estoque inicial < 0 e 95 com estoque final < 0 (`estoque_reconstruido_negativo`) | **não usar o estoque reconstruído como feature**; o sinal de ruptura deve ser tratado com cautela | estoque negativo é impossível: indica movimentações que não estão na base |
| 8 | Duplicatas e linhas de total | 0 e 0 | nada a fazer | — |
| 9 | SKUs com pouco histórico | 30 SKUs × 7 meses no máximo; SKUs lançados no meio do período têm menos | sem previsão operacional enquanto não houver meses suficientes | a regra exige 6 meses disponíveis em *t − 5* |
| 10 | Tabelas que são saída da regra atual | `baseline_cobertura`, `pedido_sugerido` | excluídas como entrada | evita vazamento e circularidade |

## 5. Data de corte

Com 7 meses, nenhuma data de corte permite avaliar a regra com H = 5.
O padrão atual (`DATA_CORTE = None`) usa os últimos 6 meses como teste (corte em 2026-03), e só se avaliam as linhas que têm informação em *t − 5*.
**Recomendação:** quando houver ≥ 17 meses completos, fixar o corte em *último mês − 6*. Até lá, as métricas reais são só diagnóstico do pipeline.

## 6. Pendências com a empresa
1. Existe histórico de vendas por código anterior a 2026-02? É o que destrava o projeto.
2. Quem define o filtro `incluir` e com qual critério?
3. As vendas já vêm líquidas de devolução?
4. Como o estoque mensal foi reconstruído, e por que fica negativo?
5. A marcação de ruptura (`possivel_ruptura`) usa qual critério?

In [ ]:
import sys; sys.path.insert(0, '..')  # raiz do projeto
# Só agregados. Não salve o notebook com esta saída se ela tiver volumes em m².
from src import carregar, preparar
df, info_carga = carregar.carregar_vendas("real")
_, info_prep = preparar.preparar_painel(df)
del df
{**info_carga, **info_prep}